# Automated E-Commerce Review Tag Generator - Kaggle Notebook (T00-T12)

This notebook implements the TODO list through **T12** with Kaggle-runnable smoke checks:

- **T00-T04**: project foundation, schemas, dataset provenance, Laptop-ACOS converter, SemEval converter
- **T05**: preprocessing, deduplication, deterministic splits, statistics
- **T06**: classical/rule baselines
- **T07**: BIO labels and token alignment
- **T08-T09**: smoke aspect extractor training, evaluation, and inference
- **T10-T12**: aspect-sentiment pair builder, smoke sentiment training, evaluation, and inference

This is designed to run immediately in Kaggle on synthetic fixtures. Full academic training/evaluation still requires the official datasets, pinned checksums, and GPU-backed transformer training.

## Kaggle Notes

Kaggle notebooks usually write persistent outputs to `/kaggle/working`. This notebook creates the project under `/kaggle/working/review_tag_generator` when running on Kaggle, or `./review_tag_generator` elsewhere.

Raw datasets and checkpoints are **not** tracked or bundled. Some datasets require manual download or license acceptance; this notebook records exact source information and validates expected paths.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import sys
from datetime import date

KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_ROOT = KAGGLE_WORKING / 'review_tag_generator' if KAGGLE_WORKING.exists() else Path.cwd() / 'review_tag_generator'
SRC_ROOT = PROJECT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'review_tag_generator'

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())

## Dependency Check

Kaggle generally includes many Python packages, but this notebook requires Pydantic v2 for schema validation. Run this cell before the schema cells.

In [ ]:
try:
    import pydantic
    major = int(pydantic.__version__.split('.')[0])
    if major < 2:
        raise ImportError(f'Pydantic v2 required, found {pydantic.__version__}')
    print('Pydantic:', pydantic.__version__)
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pydantic>=2,<3'])
    import pydantic
    print('Installed Pydantic:', pydantic.__version__)

## T00 - Repository and Project Foundation

In [ ]:
required_dirs = [
    'data/raw', 'data/interim', 'data/processed', 'data/manifests',
    'ml', 'models', 'backend', 'frontend', 'notebooks', 'tests',
    'audit/evidence/A0', 'audit/evidence/A1', 'audit/reports', 'docs',
    'src/review_tag_generator/contracts', 'scripts'
]

for rel in required_dirs:
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)

files = {
    '.python-version': '3.11\n',
    '.nvmrc': '20\n',
    '.editorconfig': '''root = true\n\n[*]\ncharset = utf-8\nend_of_line = lf\ninsert_final_newline = true\nindent_style = space\nindent_size = 4\ntrim_trailing_whitespace = true\n\n[*.{md,yml,yaml,json}]\nindent_size = 2\n''',
    '.gitignore': '''# Python\n__pycache__/\n*.py[cod]\n.venv/\nvenv/\n.pytest_cache/\n.mypy_cache/\n.ipynb_checkpoints/\n\n# Node/frontend\nnode_modules/\ndist/\nbuild/\n.vite/\n\n# Secrets and local config\n.env\n.env.*\n!.env.example\n\n# Datasets and model artifacts\ndata/raw/\ndata/interim/\ndata/processed/\nmodels/\n*.ckpt\n*.pt\n*.pth\n*.safetensors\n*.bin\n\n# Logs and generated audit evidence\n*.log\naudit/evidence/\n''',
    '.env.example': '''DATABASE_URL=\nMODEL_DIR=\nDATA_DIR=\nAPI_HOST=\nAPI_PORT=\n''',
    'backend/.env.example': '''DATABASE_URL=\nMODEL_DIR=\nCORS_ORIGINS=\n''',
    'frontend/.env.example': '''VITE_API_BASE_URL=\n''',
    'README.md': '''# Automated E-Commerce Review Tag Generator\n\nAspect-level review tagging system for laptops and consumer electronics.\n\n## Scope\n\nMVP pipeline: review text -> conservative preprocessing -> DistilBERT aspect extraction -> DistilBERT aspect-level sentiment -> MiniLM aspect normalization -> deterministic tag generation -> product aggregation -> PostgreSQL/FastAPI -> React dashboard.\n\n## Local/Kaggle Layout\n\n- `data/raw/`: user-provided raw datasets; never commit\n- `data/interim/`: converted or temporary datasets; never commit\n- `data/processed/`: model-ready splits; never commit\n- `data/manifests/`: dataset provenance and checksums\n- `models/`: checkpoints and model manifests; never commit\n- `src/review_tag_generator/contracts/`: shared schemas/config\n- `tests/`: deterministic tests and fixtures\n- `audit/`: audit register, findings, reports, and evidence\n\n## Dataset Policy\n\nDo not redistribute restricted datasets. Download Laptop-ACOS, SemEval 2014 Laptop ABSA, and Amazon Reviews 2023 from their official sources and place them under the expected `data/raw/...` paths documented in `data/manifests/dataset_manifest.json`.\n'''
}

for rel, text in files.items():
    path = PROJECT_ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')

for rel in ['src/review_tag_generator/__init__.py', 'src/review_tag_generator/contracts/__init__.py']:
    (PROJECT_ROOT / rel).write_text('', encoding='utf-8')

print('T00 files and directories created.')

## T01 - Shared Schemas and Configuration

In [ ]:
schemas_py = r'''from __future__ import annotations

from datetime import datetime
from enum import Enum
from typing import Dict, List, Optional

from pydantic import BaseModel, Field, model_validator


CONTRACT_VERSION = '0.1.0'


class SentimentLabel(str, Enum):
    positive = 'positive'
    neutral = 'neutral'
    negative = 'negative'


class BioLabel(str, Enum):
    O = 'O'
    B_ASP = 'B-ASP'
    I_ASP = 'I-ASP'


BIO_LABEL_TO_ID = {'O': 0, 'B-ASP': 1, 'I-ASP': 2}
BIO_ID_TO_LABEL = {v: k for k, v in BIO_LABEL_TO_ID.items()}
SENTIMENT_LABEL_TO_ID = {'positive': 0, 'neutral': 1, 'negative': 2}
SENTIMENT_ID_TO_LABEL = {v: k for k, v in SENTIMENT_LABEL_TO_ID.items()}


class AspectAnnotation(BaseModel):
    aspect: str = Field(min_length=1)
    normalized_aspect: Optional[str] = None
    opinion: Optional[str] = None
    sentiment: SentimentLabel
    start_char: int = Field(ge=0)
    end_char: int = Field(gt=0)

    @model_validator(mode='after')
    def validate_span(self):
        if self.end_char <= self.start_char:
            raise ValueError('end_char must be greater than start_char')
        return self


class Review(BaseModel):
    review_id: str = Field(min_length=1)
    product_id: str = Field(min_length=1)
    product_category: str = Field(min_length=1)
    review_text: str = Field(min_length=1)
    rating: Optional[int] = Field(default=None, ge=1, le=5)
    helpful_votes: Optional[int] = Field(default=None, ge=0)
    timestamp: Optional[datetime] = None
    annotations: List[AspectAnnotation] = Field(default_factory=list)

    @model_validator(mode='after')
    def validate_annotation_offsets(self):
        text_len = len(self.review_text)
        for ann in self.annotations:
            if ann.end_char > text_len:
                raise ValueError(f'annotation span exceeds review_text length: {ann.start_char}-{ann.end_char}')
        return self


class AspectPrediction(BaseModel):
    aspect: str = Field(min_length=1)
    start_char: int = Field(ge=0)
    end_char: int = Field(gt=0)
    confidence: float = Field(ge=0.0, le=1.0)

    @model_validator(mode='after')
    def validate_span(self):
        if self.end_char <= self.start_char:
            raise ValueError('end_char must be greater than start_char')
        return self


class SentimentPrediction(BaseModel):
    label: SentimentLabel
    confidence: float = Field(ge=0.0, le=1.0)
    probabilities: Dict[SentimentLabel, float]

    @model_validator(mode='after')
    def validate_probabilities(self):
        required = set(SentimentLabel)
        if set(self.probabilities) != required:
            raise ValueError('probabilities must include positive, neutral, and negative')
        total = sum(self.probabilities.values())
        if any(v < 0 or v > 1 for v in self.probabilities.values()):
            raise ValueError('probabilities must be between 0 and 1')
        if abs(total - 1.0) > 1e-6:
            raise ValueError('probabilities must sum to 1')
        return self


class NormalizationResult(BaseModel):
    raw_aspect: str = Field(min_length=1)
    normalized_aspect: Optional[str] = None
    similarity: float = Field(ge=0.0, le=1.0)
    is_unknown: bool

    @model_validator(mode='after')
    def validate_unknown_state(self):
        if self.is_unknown and self.normalized_aspect is not None:
            raise ValueError('unknown results must not include normalized_aspect')
        if not self.is_unknown and self.normalized_aspect is None:
            raise ValueError('known results must include normalized_aspect')
        return self


class TagResult(BaseModel):
    normalized_aspect: Optional[str] = None
    sentiment: SentimentLabel
    tag: str = Field(min_length=1)
    confidence: float = Field(ge=0.0, le=1.0)


class ProductTag(BaseModel):
    product_id: str = Field(min_length=1)
    normalized_aspect: str = Field(min_length=1)
    tag: str = Field(min_length=1)
    counts: Dict[SentimentLabel, int]
    ratios: Dict[SentimentLabel, float]
    mention_count: int = Field(ge=0)
    average_confidence: float = Field(ge=0.0, le=1.0)
    score: float = Field(ge=0.0)
    rank: int = Field(ge=1)


class ProcessingJob(BaseModel):
    job_id: str = Field(min_length=1)
    status: str = Field(pattern='^(queued|running|succeeded|failed|partial)$')
    created_at: datetime
    updated_at: datetime
    total_items: int = Field(default=0, ge=0)
    processed_items: int = Field(default=0, ge=0)
    failed_items: int = Field(default=0, ge=0)
    error_message: Optional[str] = None
'''

config_py = r'''from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import os


RANDOM_SEED = 42
ASPECT_MODEL_REVISION = 'distilbert-base-uncased'
SENTIMENT_MODEL_REVISION = 'distilbert-base-uncased'
NORMALIZER_MODEL_REVISION = 'sentence-transformers/all-MiniLM-L6-v2'
MAX_SEQUENCE_LENGTH = 256
NORMALIZATION_THRESHOLD = 0.72
MIN_MENTIONS_FOR_PRODUCT_TAG = 3
POSITIVE_STRONG_THRESHOLD = 0.80
POSITIVE_WEAK_THRESHOLD = 0.60
NEGATIVE_STRONG_THRESHOLD = 0.70
NEGATIVE_WEAK_THRESHOLD = 0.50


@dataclass(frozen=True)
class AppConfig:
    project_root: Path
    data_dir: Path
    model_dir: Path
    random_seed: int = RANDOM_SEED
    max_sequence_length: int = MAX_SEQUENCE_LENGTH
    normalization_threshold: float = NORMALIZATION_THRESHOLD
    min_mentions_for_product_tag: int = MIN_MENTIONS_FOR_PRODUCT_TAG


def load_config(project_root: str | Path | None = None) -> AppConfig:
    root = Path(project_root or os.getenv('REVIEW_TAG_PROJECT_ROOT', '.')).resolve()
    data_dir = Path(os.getenv('REVIEW_TAG_DATA_DIR', root / 'data')).resolve()
    model_dir = Path(os.getenv('REVIEW_TAG_MODEL_DIR', root / 'models')).resolve()
    seed = int(os.getenv('REVIEW_TAG_RANDOM_SEED', RANDOM_SEED))
    max_len = int(os.getenv('REVIEW_TAG_MAX_SEQUENCE_LENGTH', MAX_SEQUENCE_LENGTH))
    threshold = float(os.getenv('REVIEW_TAG_NORMALIZATION_THRESHOLD', NORMALIZATION_THRESHOLD))
    min_mentions = int(os.getenv('REVIEW_TAG_MIN_MENTIONS', MIN_MENTIONS_FOR_PRODUCT_TAG))
    return AppConfig(root, data_dir, model_dir, seed, max_len, threshold, min_mentions)
'''

(PACKAGE_ROOT / 'contracts' / 'schemas.py').write_text(schemas_py, encoding='utf-8')
(PACKAGE_ROOT / 'contracts' / 'config.py').write_text(config_py, encoding='utf-8')

print('T01 schema and config modules written.')

In [ ]:
sys.path.insert(0, str(SRC_ROOT))

from pydantic import ValidationError
from review_tag_generator.contracts.config import load_config
from review_tag_generator.contracts.schemas import (
    BIO_LABEL_TO_ID,
    SENTIMENT_LABEL_TO_ID,
    AspectAnnotation,
    AspectPrediction,
    NormalizationResult,
    Review,
    SentimentLabel,
    SentimentPrediction,
)

example = Review(
    review_id='R123',
    product_id='P001',
    product_category='Laptop',
    review_text='Excellent screen but poor battery life.',
    rating=3,
    helpful_votes=5,
    annotations=[
        AspectAnnotation(aspect='screen', normalized_aspect='display_quality', opinion='Excellent', sentiment='positive', start_char=10, end_char=16),
        AspectAnnotation(aspect='battery life', normalized_aspect='battery_life', opinion='poor', sentiment='negative', start_char=26, end_char=38),
    ],
)

round_trip = Review.model_validate_json(example.model_dump_json())
assert round_trip == example
assert BIO_LABEL_TO_ID == {'O': 0, 'B-ASP': 1, 'I-ASP': 2}
assert SENTIMENT_LABEL_TO_ID == {'positive': 0, 'neutral': 1, 'negative': 2}
assert load_config(PROJECT_ROOT).random_seed == 42

try:
    Review(review_id='bad', product_id='P001', product_category='Laptop', review_text='Short text', rating=6)
    raise AssertionError('invalid rating was accepted')
except ValidationError:
    pass

try:
    Review(
        review_id='bad-span', product_id='P001', product_category='Laptop', review_text='Short text',
        annotations=[AspectAnnotation(aspect='battery', sentiment='positive', start_char=0, end_char=99)]
    )
    raise AssertionError('invalid offset was accepted')
except ValidationError:
    pass

SentimentPrediction(label=SentimentLabel.positive, confidence=0.9, probabilities={
    SentimentLabel.positive: 0.9,
    SentimentLabel.neutral: 0.05,
    SentimentLabel.negative: 0.05,
})
NormalizationResult(raw_aspect='battery backup', normalized_aspect='battery_life', similarity=0.84, is_unknown=False)

print('T01 validation checks passed.')

## T02 - Dataset Acquisition and Provenance

This section creates a reproducible dataset manifest and validation helper. It does not redistribute raw data.

## Official Dataset Paths

Verified source paths used by this notebook:

- Laptop-ACOS: `https://github.com/NUSTM/ACOS/tree/main/data/Laptop-ACOS` with `laptop_quad_train.tsv`, `laptop_quad_dev.tsv`, and `laptop_quad_test.tsv`.
- SemEval 2014 Task 4 Laptop ABSA: official Data and Tools page at `https://alt.qcri.org/semeval2014/task4/index.php?id=data-and-tools`; download train V2.0 and Phase B gold test data, then place `Laptop_Train.xml` and `Laptops_Test_Gold.xml` under `data/raw/semeval_2014_laptop/`.
- Amazon Reviews 2023 Electronics: `https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz` and `https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz`.


In [ ]:
manifest = {
    'manifest_version': '0.1.0',
    'retrieval_date': date.today().isoformat(),
    'datasets': [
        {
            'id': 'laptop_acos',
            'name': 'Laptop-ACOS',
            'purpose': 'primary supervised aspect/opinion/sentiment training',
            'official_source_url': 'https://github.com/NUSTM/ACOS',
            'version_or_revision': 'pin the Git commit used after download',
            'license_or_usage_note': 'Check upstream repository and cited paper terms before use; do not redistribute raw data unless license permits.',
            'manual_acceptance_required': False,
            'expected_raw_path': 'data/raw/laptop_acos/',
            'expected_files': ['laptop_quad_train.tsv', 'laptop_quad_dev.tsv', 'laptop_quad_test.tsv'],
            'official_raw_urls': [
                'https://raw.githubusercontent.com/NUSTM/ACOS/main/data/Laptop-ACOS/laptop_quad_train.tsv',
                'https://raw.githubusercontent.com/NUSTM/ACOS/main/data/Laptop-ACOS/laptop_quad_dev.tsv',
                'https://raw.githubusercontent.com/NUSTM/ACOS/main/data/Laptop-ACOS/laptop_quad_test.tsv',
            ],
            'checksum_sha256': None,
        },
        {
            'id': 'semeval_2014_laptop_absa',
            'name': 'SemEval 2014 Task 4 Laptop ABSA',
            'purpose': 'benchmark and additional supervised ABSA data',
            'official_source_url': 'https://alt.qcri.org/semeval2014/task4/index.php?id=data-and-tools',
            'version_or_revision': 'official SemEval 2014 Task 4 laptop train data V2.0 and Phase B gold test data',
            'license_or_usage_note': 'Usually requires task data access from official SemEval/organizer distribution; preserve train/test boundary.',
            'manual_acceptance_required': True,
            'expected_raw_path': 'data/raw/semeval_2014_laptop/',
            'expected_files': ['Laptop_Train.xml', 'Laptops_Test_Gold.xml'],
            'official_download_note': 'Download via the official SemEval Task 4 Data and Tools page / MetaShare links; file names may be inside released zip archives.',
            'checksum_sha256': None,
        },
        {
            'id': 'amazon_reviews_2023_sample',
            'name': 'Amazon Reviews 2023 bounded electronics/laptop sample',
            'purpose': 'inference-only product aggregation and UI demonstration',
            'official_source_url': 'https://amazon-reviews-2023.github.io/main.html',
            'version_or_revision': 'record exact category file and revision/date used',
            'license_or_usage_note': 'Use a bounded 5,000-50,000 review sample; exclude reviewer PII; do not train supervised aspect labels from this dataset by default.',
            'manual_acceptance_required': True,
            'expected_raw_path': 'data/raw/amazon_reviews_2023/',
            'expected_files': ['Electronics.jsonl.gz', 'meta_Electronics.jsonl.gz'],
            'official_raw_urls': [
                'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz',
                'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz',
            ],
            'checksum_sha256': None,
        },
    ],
}

manifest_path = PROJECT_ROOT / 'data/manifests/dataset_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print('Wrote manifest:', manifest_path)

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def validate_dataset_manifest(manifest_path: Path, project_root: Path, dry_run: bool = True):
    manifest_data = json.loads(manifest_path.read_text(encoding='utf-8'))
    results = []
    for dataset in manifest_data['datasets']:
        raw_dir = project_root / dataset['expected_raw_path']
        expected_files = dataset.get('expected_files') or []
        missing_files = [name for name in expected_files if not (raw_dir / name).exists()]
        status = 'ready'
        action = 'No action required.'
        if missing_files:
            status = 'missing_manual_download' if dataset.get('manual_acceptance_required') else 'missing_download'
            action = f"Place required files in {raw_dir}: {missing_files}"
        elif not raw_dir.exists():
            status = 'raw_path_missing'
            action = f"Create or populate {raw_dir}."
        results.append({
            'dataset_id': dataset['id'],
            'status': status,
            'source': dataset['official_source_url'],
            'manual_acceptance_required': dataset['manual_acceptance_required'],
            'expected_raw_path': str(raw_dir),
            'missing_files': missing_files,
            'action': action,
        })
    return results


validation_results = validate_dataset_manifest(manifest_path, PROJECT_ROOT, dry_run=True)
print(json.dumps(validation_results, indent=2))

assert any(r['dataset_id'] == 'semeval_2014_laptop_absa' and r['manual_acceptance_required'] for r in validation_results)
print('T02 dry-run validation completed. Missing/manual datasets are expected until you add raw files.')

## T03 - Laptop-ACOS Converter

This section adds a structural Laptop-ACOS converter. It handles the official ACOS quadruple-style lines, writes common-schema JSONL, logs rejected lines/annotations, and proves deterministic behavior with fixtures.

Important policy: implicit aspects use `-1,-1` token spans in Laptop-ACOS, but the current T01 schema requires honest character offsets. This converter rejects implicit-aspect annotations with `implicit_aspect_unsupported` instead of fabricating offsets.

In [ ]:
import csv
from collections import Counter, defaultdict
from typing import Any


def stable_id(*parts: object) -> str:
    raw = '::'.join(str(part) for part in parts)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:16]


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def write_jsonl(path: Path, rows: list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for row in rows:
        if hasattr(row, 'model_dump'):
            payload = row.model_dump(mode='json')
        else:
            payload = row
        lines.append(json.dumps(payload, sort_keys=True, separators=(',', ':')))
    path.write_text(('\n'.join(lines) + ('\n' if lines else '')), encoding='utf-8')


def parse_token_span(span: str):
    try:
        start_s, end_s = span.split(',', 1)
        start, end = int(start_s), int(end_s)
    except Exception as exc:
        raise ValueError(f'invalid span format: {span!r}') from exc
    if start == -1 and end == -1:
        return None
    if start < 0 or end <= start:
        raise ValueError(f'invalid token span bounds: {span!r}')
    return start, end


def token_char_offsets(tokens: list[str]) -> tuple[str, list[tuple[int, int]]]:
    text_parts = []
    offsets = []
    cursor = 0
    for token in tokens:
        if text_parts:
            text_parts.append(' ')
            cursor += 1
        start = cursor
        text_parts.append(token)
        cursor += len(token)
        offsets.append((start, cursor))
    return ''.join(text_parts), offsets


def find_quad_start(tokens: list[str]) -> int:
    for i in range(len(tokens)):
        remaining = len(tokens) - i
        if remaining >= 4 and remaining % 4 == 0:
            ok = True
            for j in range(i, len(tokens), 4):
                try:
                    parse_token_span(tokens[j])
                    if '#' not in tokens[j + 1]:
                        ok = False
                        break
                    if tokens[j + 2] not in {'0', '1', '2'}:
                        ok = False
                        break
                    parse_token_span(tokens[j + 3])
                except Exception:
                    ok = False
                    break
            if ok:
                return i
    raise ValueError('no_quadruples')


def laptop_acos_rejection(split: str, source_file: str, line_number: int, raw_line: str, reason_code: str, reason_detail: str, stage: str) -> dict[str, Any]:
    return {
        'dataset_id': 'laptop_acos',
        'split': split,
        'source_file': source_file,
        'line_number': line_number,
        'raw_line_sha256': sha256_text(raw_line),
        'reason_code': reason_code,
        'reason_detail': reason_detail,
        'raw_line_preview': raw_line[:240],
        'stage': stage,
    }


def parse_laptop_acos_line(raw_line: str, split: str, line_number: int, source_file: str):
    raw_line = raw_line.rstrip('\n')
    rejects = []
    if not raw_line.strip():
        return None, [laptop_acos_rejection(split, source_file, line_number, raw_line, 'empty_line', 'line is empty', 'parse')]
    tokens = raw_line.split()
    try:
        quad_start = find_quad_start(tokens)
    except Exception as exc:
        return None, [laptop_acos_rejection(split, source_file, line_number, raw_line, 'no_quadruples', str(exc), 'parse')]

    text_tokens = tokens[:quad_start]
    quad_tokens = tokens[quad_start:]
    if not text_tokens:
        return None, [laptop_acos_rejection(split, source_file, line_number, raw_line, 'missing_text', 'no review text before quadruples', 'parse')]
    text, offsets = token_char_offsets(text_tokens)

    annotations = []
    for q_index in range(0, len(quad_tokens), 4):
        aspect_span_s, category, sentiment_s, opinion_span_s = quad_tokens[q_index:q_index + 4]
        try:
            if '#' not in category:
                raise ValueError(f'invalid category: {category!r}')
            sentiment = {'0': 'negative', '1': 'neutral', '2': 'positive'}[sentiment_s]
            aspect_span = parse_token_span(aspect_span_s)
            opinion_span = parse_token_span(opinion_span_s)
            if aspect_span is None:
                rejects.append(laptop_acos_rejection(split, source_file, line_number, raw_line, 'implicit_aspect_unsupported', 'T01 schema requires concrete character offsets', 'span'))
                continue
            start_tok, end_tok = aspect_span
            if end_tok > len(offsets):
                raise ValueError(f'aspect span out of bounds: {aspect_span_s}')
            start_char = offsets[start_tok][0]
            end_char = offsets[end_tok - 1][1]
            aspect = ' '.join(text_tokens[start_tok:end_tok])
            opinion = None
            if opinion_span is not None:
                op_start, op_end = opinion_span
                if op_end > len(offsets):
                    raise ValueError(f'opinion span out of bounds: {opinion_span_s}')
                opinion = ' '.join(text_tokens[op_start:op_end])
            annotations.append(AspectAnnotation(
                aspect=aspect,
                normalized_aspect=category,
                opinion=opinion,
                sentiment=sentiment,
                start_char=start_char,
                end_char=end_char,
            ))
        except KeyError:
            rejects.append(laptop_acos_rejection(split, source_file, line_number, raw_line, 'invalid_sentiment', sentiment_s, 'parse'))
        except Exception as exc:
            rejects.append(laptop_acos_rejection(split, source_file, line_number, raw_line, 'malformed_quadruple', str(exc), 'parse'))

    try:
        review = Review(
            review_id=f'laptop_acos:{split}:{line_number:06d}:{sha256_text(raw_line)[:12]}',
            product_id='laptop_acos',
            product_category='Laptop',
            review_text=text,
            rating=None,
            helpful_votes=None,
            annotations=annotations,
        )
    except Exception as exc:
        return None, rejects + [laptop_acos_rejection(split, source_file, line_number, raw_line, 'schema_validation_failed', str(exc), 'schema_validation')]
    return review, rejects


def discover_laptop_acos_files(raw_dir: Path) -> tuple[dict[str, Path], list[str]]:
    names = {'train': 'laptop_quad_train.tsv', 'dev': 'laptop_quad_dev.tsv', 'test': 'laptop_quad_test.tsv'}
    found = {}
    missing = []
    for split, filename in names.items():
        direct = raw_dir / filename
        nested = raw_dir / 'data' / 'Laptop-ACOS' / filename
        if direct.exists():
            found[split] = direct
        elif nested.exists():
            found[split] = nested
        else:
            matches = sorted(raw_dir.rglob(filename)) if raw_dir.exists() else []
            if len(matches) == 1:
                found[split] = matches[0]
            else:
                missing.append(filename)
    return found, missing


def convert_laptop_acos(raw_dir: Path, output_dir: Path, dry_run: bool = False) -> dict[str, Any]:
    files, missing = discover_laptop_acos_files(raw_dir)
    if missing:
        return {
            'status': 'missing_raw_files',
            'raw_dir': str(raw_dir),
            'missing_files': missing,
            'expected_files': ['laptop_quad_train.tsv', 'laptop_quad_dev.tsv', 'laptop_quad_test.tsv'],
            'manual_step': f'Place Laptop-ACOS TSV files under {raw_dir} or clone the ACOS repo there.',
        }

    reviews = []
    rejects = []
    split_counts = Counter()
    sentiment_counts = Counter()
    category_counts = Counter()
    raw_hashes = {}
    for split in ['train', 'dev', 'test']:
        path = files[split]
        raw_hashes[str(path)] = sha256_file(path)
        for line_number, raw_line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            review, line_rejects = parse_laptop_acos_line(raw_line, split, line_number, str(path))
            rejects.extend(line_rejects)
            if review is not None:
                reviews.append(review)
                split_counts[split] += 1
                for ann in review.annotations:
                    sentiment_counts[ann.sentiment.value] += 1
                    if ann.normalized_aspect:
                        category_counts[ann.normalized_aspect] += 1

    out_path = output_dir / 'laptop_acos_common.jsonl'
    rejected_path = output_dir / 'laptop_acos_rejected.jsonl'
    report_path = PROJECT_ROOT / 'audit/evidence/A1/laptop_acos_conversion_summary.json'
    report = {
        'dataset_id': 'laptop_acos',
        'converter_version': '0.1.0',
        'status': 'ok',
        'raw_file_sha256': raw_hashes,
        'emitted_reviews': len(reviews),
        'rejected_items': len(rejects),
        'split_counts': dict(split_counts),
        'annotation_count': sum(len(r.annotations) for r in reviews),
        'sentiment_distribution': dict(sentiment_counts),
        'category_count': dict(category_counts),
        'output_jsonl': str(out_path),
        'rejected_jsonl': str(rejected_path),
    }
    if not dry_run:
        write_jsonl(out_path, reviews)
        write_jsonl(rejected_path, rejects)
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report['output_sha256'] = sha256_file(out_path)
        report['rejected_sha256'] = sha256_file(rejected_path)
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return report

print('T03 Laptop-ACOS converter functions loaded.')

In [ ]:
fixture_lines = [
    'screen is good . 0,1 DISPLAY#GENERAL 2 2,3',
    'battery life is long but keyboard feels cheap . 0,2 BATTERY#OPERATION_PERFORMANCE 2 3,4 5,6 KEYBOARD#QUALITY 0 7,8',
    'works great ! -1,-1 LAPTOP#GENERAL 2 1,2',
    'one day it just would not power up . 6,8 BATTERY#OPERATION_PERFORMANCE 0 -1,-1',
    'bad sentiment here 0,1 DISPLAY#GENERAL 9 1,2',
    'bad span here 0,9 DISPLAY#GENERAL 2 1,2',
]

fixture_dir = PROJECT_ROOT / 'tests/fixtures/laptop_acos'
fixture_dir.mkdir(parents=True, exist_ok=True)
for split in ['train', 'dev', 'test']:
    (fixture_dir / f'laptop_quad_{split}.tsv').write_text('\n'.join(fixture_lines) + '\n', encoding='utf-8')

acos_result_1 = convert_laptop_acos(fixture_dir, PROJECT_ROOT / 'data/interim/laptop_acos')
acos_bytes_1 = (PROJECT_ROOT / 'data/interim/laptop_acos/laptop_acos_common.jsonl').read_bytes()
acos_rejects_1 = (PROJECT_ROOT / 'data/interim/laptop_acos/laptop_acos_rejected.jsonl').read_text(encoding='utf-8')
acos_result_2 = convert_laptop_acos(fixture_dir, PROJECT_ROOT / 'data/interim/laptop_acos')
assert acos_result_1['emitted_reviews'] == acos_result_2['emitted_reviews']
assert acos_bytes_1 == (PROJECT_ROOT / 'data/interim/laptop_acos/laptop_acos_common.jsonl').read_bytes()
assert acos_result_1['emitted_reviews'] == 15
assert acos_result_1['annotation_count'] == 12
assert 'implicit_aspect_unsupported' in acos_rejects_1
assert 'bad span here' in acos_rejects_1
assert 'no_quadruples' in acos_rejects_1
rows = [Review.model_validate_json(line) for line in (PROJECT_ROOT / 'data/interim/laptop_acos/laptop_acos_common.jsonl').read_text(encoding='utf-8').splitlines()]
assert rows[0].review_text[0:6] == 'screen'
assert rows[0].annotations[0].sentiment.value == 'positive'
assert rows[1].annotations[0].aspect == 'battery life'
assert rows[1].annotations[1].aspect == 'keyboard'

real_acos_result = convert_laptop_acos(PROJECT_ROOT / 'data/raw/laptop_acos', PROJECT_ROOT / 'data/interim/laptop_acos_real', dry_run=True)
print('T03 fixture result:', {k: acos_result_1[k] for k in ['status', 'emitted_reviews', 'rejected_items', 'annotation_count']})
print('T03 real raw dry-run:', real_acos_result)

## T04 - SemEval 2014 Laptop ABSA Converter

This section adds a SemEval converter using `xml.etree.ElementTree`, preserving official train/test boundaries. Sentences without aspects are emitted with `annotations=[]`. `conflict` polarity is skipped explicitly, not coerced to neutral. Bad spans are logged instead of silently repaired.

In [ ]:
import xml.etree.ElementTree as ET


def normalize_semeval_sentiment(value: str) -> str:
    label = str(value).strip().lower()
    if label not in {'positive', 'neutral', 'negative'}:
        raise ValueError(f'unsupported polarity: {value!r}')
    return label


def semeval_rejection(split: str, source_file: str, sentence_id: str, reason_code: str, reason_detail: str, **extra) -> dict[str, Any]:
    payload = {
        'dataset_id': 'semeval_2014_laptop_absa',
        'split': split,
        'source_file': source_file,
        'sentence_id': sentence_id,
        'reason_code': reason_code,
        'reason_detail': reason_detail,
    }
    payload.update(extra)
    return payload


def parse_semeval_xml_file(path: Path, split: str) -> tuple[list[Review], list[dict[str, Any]], dict[str, Any]]:
    root = ET.parse(path).getroot()
    reviews = []
    rejects = []
    stats = Counter()
    for sentence_index, sentence in enumerate(root.findall('.//sentence'), start=1):
        sentence_id = sentence.attrib.get('id') or f'missing-id-{sentence_index}'
        text_el = sentence.find('text')
        text = (text_el.text or '').strip() if text_el is not None else ''
        if not text:
            rejects.append(semeval_rejection(split, str(path), sentence_id, 'missing_text', 'sentence text is empty'))
            continue
        annotations = []
        aspect_terms = sentence.find('aspectTerms')
        if aspect_terms is None or not list(aspect_terms):
            stats['sentences_without_aspects'] += 1
        else:
            for term in aspect_terms.findall('aspectTerm'):
                aspect = term.attrib.get('term', '').strip()
                polarity = term.attrib.get('polarity', '').strip().lower()
                if not aspect:
                    rejects.append(semeval_rejection(split, str(path), sentence_id, 'missing_term', 'aspect term is empty'))
                    continue
                if polarity == 'conflict':
                    stats['conflict_annotations_skipped'] += 1
                    rejects.append(semeval_rejection(split, str(path), sentence_id, 'conflict_polarity', 'conflict polarity is not mapped', term=aspect))
                    continue
                try:
                    sentiment = normalize_semeval_sentiment(polarity)
                    start = int(term.attrib['from'])
                    end = int(term.attrib['to'])
                    if not (0 <= start < end <= len(text)):
                        raise ValueError(f'offset out of bounds: {start}-{end}')
                    observed = text[start:end]
                    if observed != aspect:
                        raise ValueError(f'span substring {observed!r} does not equal term {aspect!r}')
                    annotations.append(AspectAnnotation(
                        aspect=aspect,
                        normalized_aspect=None,
                        opinion=None,
                        sentiment=sentiment,
                        start_char=start,
                        end_char=end,
                    ))
                except KeyError as exc:
                    rejects.append(semeval_rejection(split, str(path), sentence_id, 'missing_offset', str(exc), term=aspect))
                except Exception as exc:
                    rejects.append(semeval_rejection(split, str(path), sentence_id, 'invalid_annotation', str(exc), term=aspect))
        reviews.append(Review(
            review_id=f'semeval2014_laptop:{split}:{sentence_id}',
            product_id='semeval2014_laptop',
            product_category='Laptop',
            review_text=text,
            rating=None,
            helpful_votes=None,
            annotations=annotations,
        ))
        stats['sentences'] += 1
        stats['annotations_emitted'] += len(annotations)
    return reviews, rejects, dict(stats)


def convert_semeval_laptop(raw_dir: Path, output_dir: Path, dry_run: bool = False) -> dict[str, Any]:
    expected = {'train': 'Laptop_Train.xml', 'test': 'Laptops_Test_Gold.xml'}
    missing = [name for name in expected.values() if not (raw_dir / name).exists()]
    if missing:
        return {
            'status': 'missing_manual_download',
            'raw_dir': str(raw_dir),
            'missing_files': missing,
            'manual_step': f'Place official SemEval laptop XML files under {raw_dir}.',
        }

    output_dir.mkdir(parents=True, exist_ok=True)
    all_rejects = []
    report = {
        'dataset_id': 'semeval_2014_laptop_absa',
        'converter_version': '0.1.0',
        'status': 'ok',
        'splits': {},
    }
    for split, filename in expected.items():
        path = raw_dir / filename
        reviews, rejects, stats = parse_semeval_xml_file(path, split)
        all_rejects.extend(rejects)
        split_out = output_dir / f'{split}.jsonl'
        if not dry_run:
            write_jsonl(split_out, reviews)
        report['splits'][split] = {
            'input_file': str(path),
            'input_sha256': sha256_file(path),
            'output_jsonl': str(split_out),
            'emitted_reviews': len(reviews),
            'rejected_annotations': len(rejects),
            **stats,
        }
    rejected_path = output_dir / 'semeval_2014_laptop_rejected.jsonl'
    report_path = PROJECT_ROOT / 'audit/evidence/A1/semeval_2014_laptop_conversion_summary.json'
    report['total_rejected'] = len(all_rejects)
    if not dry_run:
        write_jsonl(rejected_path, all_rejects)
        report['rejected_jsonl'] = str(rejected_path)
        report['rejected_sha256'] = sha256_file(rejected_path)
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return report

print('T04 SemEval converter functions loaded.')

In [ ]:
semeval_fixture_dir = PROJECT_ROOT / 'tests/fixtures/semeval_2014_laptop'
semeval_fixture_dir.mkdir(parents=True, exist_ok=True)
train_xml = '''<sentences>
  <sentence id="1">
    <text>Great screen, but the battery is weak.</text>
    <aspectTerms>
      <aspectTerm term="screen" polarity="positive" from="6" to="12"/>
      <aspectTerm term="battery" polarity="negative" from="22" to="29"/>
    </aspectTerms>
  </sentence>
  <sentence id="2">
    <text>No aspect here &amp; still valid.</text>
  </sentence>
  <sentence id="3">
    <text>The keyboard is okay.</text>
    <aspectTerms>
      <aspectTerm term="keyboard" polarity="conflict" from="4" to="12"/>
    </aspectTerms>
  </sentence>
</sentences>'''

test_xml = '''<sentences>
  <sentence id="4">
    <text>Fast processor and sharp display.</text>
    <aspectTerms>
      <aspectTerm term="processor" polarity="positive" from="5" to="14"/>
      <aspectTerm term="display" polarity="positive" from="25" to="32"/>
    </aspectTerms>
  </sentence>
  <sentence id="5">
    <text>Ports are useful.</text>
    <aspectTerms>
      <aspectTerm term="Ports" polarity="neutral" from="0" to="5"/>
    </aspectTerms>
  </sentence>
  <sentence id="bad">
    <text>Camera is blurry.</text>
    <aspectTerms>
      <aspectTerm term="Camera" polarity="negative" from="1" to="7"/>
    </aspectTerms>
  </sentence>
</sentences>'''

(semeval_fixture_dir / 'Laptop_Train.xml').write_text(train_xml, encoding='utf-8')
(semeval_fixture_dir / 'Laptops_Test_Gold.xml').write_text(test_xml, encoding='utf-8')

semeval_output_dir = PROJECT_ROOT / 'data/interim/semeval_2014_laptop'
semeval_result_1 = convert_semeval_laptop(semeval_fixture_dir, semeval_output_dir)
train_bytes_1 = (semeval_output_dir / 'train.jsonl').read_bytes()
test_bytes_1 = (semeval_output_dir / 'test.jsonl').read_bytes()
semeval_result_2 = convert_semeval_laptop(semeval_fixture_dir, semeval_output_dir)
assert semeval_result_1['splits']['train']['emitted_reviews'] == 3
assert semeval_result_1['splits']['test']['emitted_reviews'] == 3
assert semeval_result_1['total_rejected'] == 2
assert train_bytes_1 == (semeval_output_dir / 'train.jsonl').read_bytes()
assert test_bytes_1 == (semeval_output_dir / 'test.jsonl').read_bytes()
train_rows = [Review.model_validate_json(line) for line in (semeval_output_dir / 'train.jsonl').read_text(encoding='utf-8').splitlines()]
test_rows = [Review.model_validate_json(line) for line in (semeval_output_dir / 'test.jsonl').read_text(encoding='utf-8').splitlines()]
assert train_rows[0].review_text[6:12] == 'screen'
assert train_rows[0].review_text[22:29] == 'battery'
assert train_rows[1].review_text == 'No aspect here & still valid.'
assert train_rows[1].annotations == []
assert all(row.review_id.startswith('semeval2014_laptop:train:') for row in train_rows)
assert all(row.review_id.startswith('semeval2014_laptop:test:') for row in test_rows)
rejected_text = (semeval_output_dir / 'semeval_2014_laptop_rejected.jsonl').read_text(encoding='utf-8')
assert 'conflict_polarity' in rejected_text
assert 'span substring' in rejected_text

real_semeval_result = convert_semeval_laptop(PROJECT_ROOT / 'data/raw/semeval_2014_laptop', PROJECT_ROOT / 'data/interim/semeval_2014_laptop_real', dry_run=True)
print('T04 fixture result:', {'status': semeval_result_1['status'], 'train': semeval_result_1['splits']['train'], 'test': semeval_result_1['splits']['test'], 'total_rejected': semeval_result_1['total_rejected']})
print('T04 real raw dry-run:', real_semeval_result)

## T05 - Preprocessing, Deduplication, Splits, and Statistics

This smoke implementation keeps transformer-safe text: it removes HTML tags, normalizes whitespace, optionally removes URLs/emails, preserves punctuation and negation, remaps character offsets, removes exact duplicates, keeps official SemEval split IDs, and deterministically splits other records.

In [ ]:
import html


def clean_text_with_mapping(text, remove_urls=False, remove_emails=False):
    skip = [False] * len(text)
    for match in re.finditer(r'<[^>]+>', text):
        for i in range(match.start(), match.end()):
            skip[i] = True
    if remove_urls:
        for match in re.finditer(r'https?://\S+|www\.\S+', text):
            for i in range(match.start(), match.end()):
                skip[i] = True
    if remove_emails:
        for match in re.finditer(r'\b[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}\b', text):
            for i in range(match.start(), match.end()):
                skip[i] = True

    out = []
    old_to_new = [None] * len(text)
    pending_space_old = []
    for i, ch in enumerate(text):
        if skip[i]:
            continue
        if ch.isspace():
            pending_space_old.append(i)
            continue
        if pending_space_old and out and out[-1] != ' ':
            new_index = len(out)
            out.append(' ')
            for old_i in pending_space_old:
                old_to_new[old_i] = new_index
        pending_space_old = []
        new_index = len(out)
        out.append(ch)
        old_to_new[i] = new_index
    cleaned = ''.join(out).strip()
    leading_trim = len(''.join(out)) - len(''.join(out).lstrip())
    if leading_trim:
        cleaned = cleaned.lstrip()
        old_to_new = [None if v is None else max(0, v - leading_trim) for v in old_to_new]
    return cleaned, old_to_new


def remap_annotation(annotation, old_text, new_text, old_to_new):
    mapped = [old_to_new[i] for i in range(annotation.start_char, annotation.end_char) if i < len(old_to_new) and old_to_new[i] is not None]
    if not mapped:
        raise ValueError('annotation removed during cleaning')
    start, end = min(mapped), max(mapped) + 1
    if end > len(new_text):
        raise ValueError('remapped span out of bounds')
    observed = new_text[start:end]
    if observed.lower() != annotation.aspect.lower():
        raise ValueError(f'remapped span mismatch: expected {annotation.aspect!r}, observed {observed!r}')
    return AspectAnnotation(
        aspect=annotation.aspect,
        normalized_aspect=annotation.normalized_aspect,
        opinion=annotation.opinion,
        sentiment=annotation.sentiment,
        start_char=start,
        end_char=end,
    )


def preprocess_review(review):
    cleaned, mapping = clean_text_with_mapping(review.review_text, remove_urls=True, remove_emails=True)
    if len(cleaned.strip()) < 3:
        raise ValueError('empty_or_short_review')
    anns = [remap_annotation(ann, review.review_text, cleaned, mapping) for ann in review.annotations]
    return Review(
        review_id=review.review_id,
        product_id=review.product_id,
        product_category=review.product_category,
        review_text=cleaned,
        rating=review.rating,
        helpful_votes=review.helpful_votes,
        timestamp=review.timestamp,
        annotations=anns,
    )


def load_fixture_reviews_for_t05():
    rows = []
    for path in [
        PROJECT_ROOT / 'data/interim/laptop_acos/laptop_acos_common.jsonl',
        PROJECT_ROOT / 'data/interim/semeval_2014_laptop/train.jsonl',
        PROJECT_ROOT / 'data/interim/semeval_2014_laptop/test.jsonl',
    ]:
        if path.exists():
            rows.extend(Review.model_validate_json(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip())
    rows.extend([
        Review(review_id='smoke:train:1', product_id='smoke', product_category='Laptop', review_text='Fast processor and sharp display.', annotations=[
            AspectAnnotation(aspect='processor', sentiment='positive', start_char=5, end_char=14),
            AspectAnnotation(aspect='display', sentiment='positive', start_char=25, end_char=32),
        ]),
        Review(review_id='smoke:train:2', product_id='smoke', product_category='Laptop', review_text='Ports are useful and the camera is sharp.', annotations=[
            AspectAnnotation(aspect='Ports', sentiment='neutral', start_char=0, end_char=5),
            AspectAnnotation(aspect='camera', sentiment='positive', start_char=25, end_char=31),
        ]),
        Review(review_id='smoke:train:3', product_id='smoke', product_category='Laptop', review_text='good camera but poor battery', annotations=[
            AspectAnnotation(aspect='camera', sentiment='positive', start_char=5, end_char=11),
            AspectAnnotation(aspect='battery', sentiment='negative', start_char=21, end_char=28),
        ]),
        Review(review_id='smoke:test:1', product_id='smoke', product_category='Laptop', review_text='great camera but awful battery', annotations=[
            AspectAnnotation(aspect='camera', sentiment='positive', start_char=6, end_char=12),
            AspectAnnotation(aspect='battery', sentiment='negative', start_char=23, end_char=30),
        ]),
    ])
    return rows


def split_name(review):
    rid = review.review_id
    if ':train:' in rid:
        return 'train'
    if ':dev:' in rid:
        return 'dev'
    if ':test:' in rid:
        return 'test'
    bucket = int(hashlib.sha256(rid.encode('utf-8')).hexdigest()[:8], 16) % 10
    if bucket < 8:
        return 'train'
    if bucket == 8:
        return 'dev'
    return 'test'


def preprocess_dedup_split(reviews):
    processed = []
    rejected = []
    seen = set()
    for review in reviews:
        try:
            item = preprocess_review(review)
            key = (item.product_id, item.review_text)
            if key in seen:
                rejected.append({'review_id': review.review_id, 'reason': 'exact_duplicate'})
                continue
            seen.add(key)
            processed.append(item)
        except Exception as exc:
            rejected.append({'review_id': review.review_id, 'reason': str(exc)})
    splits = {'train': [], 'dev': [], 'test': []}
    for item in processed:
        splits[split_name(item)].append(item)
    stats = {
        'input_reviews': len(reviews),
        'processed_reviews': len(processed),
        'rejected_reviews': len(rejected),
        'split_counts': {k: len(v) for k, v in splits.items()},
        'annotation_count': sum(len(r.annotations) for r in processed),
        'sentiment_counts': dict(Counter(ann.sentiment.value for r in processed for ann in r.annotations)),
        'aspect_counts': dict(Counter(ann.aspect.lower() for r in processed for ann in r.annotations)),
    }
    return splits, rejected, stats

fixture_reviews_t05 = load_fixture_reviews_for_t05()
splits_t05, rejected_t05, stats_t05 = preprocess_dedup_split(fixture_reviews_t05)

cleaned, _ = clean_text_with_mapping('<p>The camera isn\'t good.</p> Visit https://example.com', remove_urls=True)
assert cleaned == "The camera isn't good. Visit"
neg_review = Review(review_id='neg', product_id='p', product_category='Laptop', review_text="The camera isn't good.", annotations=[AspectAnnotation(aspect='camera', sentiment='negative', start_char=4, end_char=10)])
assert preprocess_review(neg_review).annotations[0].start_char == 4
assert stats_t05['processed_reviews'] > 0
assert stats_t05['split_counts']['train'] > 0

processed_dir = PROJECT_ROOT / 'data/processed/smoke'
processed_dir.mkdir(parents=True, exist_ok=True)
for split, rows in splits_t05.items():
    write_jsonl(processed_dir / f'{split}.jsonl', rows)
(processed_dir / 'preprocessing_stats.json').write_text(json.dumps(stats_t05, indent=2, sort_keys=True) + '\n', encoding='utf-8')
write_jsonl(processed_dir / 'preprocessing_rejected.jsonl', rejected_t05)
print('T05 stats:', stats_t05)

## T06 - Classical Baselines

For Kaggle smoke testing, this section provides deterministic non-neural baselines: a lexicon whole-review sentiment classifier and a rule-based aspect extractor. If scikit-learn is available later, a TF-IDF/logistic baseline can be added without changing downstream contracts.

In [ ]:
ASPECT_TERMS = [
    'battery life', 'battery', 'screen', 'display', 'keyboard', 'processor', 'camera', 'ports', 'performance', 'heating', 'price', 'speaker', 'trackpad', 'build quality'
]
POSITIVE_WORDS = {'good', 'great', 'excellent', 'bright', 'fast', 'sharp', 'long', 'useful', 'amazing', 'forever'}
NEGATIVE_WORDS = {'bad', 'poor', 'weak', 'cheap', 'terrible', 'awful', 'blurry', 'slow', 'hot', 'overheats'}


def rule_extract_aspects(text):
    results = []
    occupied = []
    for term in sorted(ASPECT_TERMS, key=len, reverse=True):
        for match in re.finditer(r'(?<!\w)' + re.escape(term) + r'(?!\w)', text, flags=re.IGNORECASE):
            span = (match.start(), match.end())
            if any(not (span[1] <= s or span[0] >= e) for s, e in occupied):
                continue
            occupied.append(span)
            results.append(AspectPrediction(aspect=text[span[0]:span[1]], start_char=span[0], end_char=span[1], confidence=0.65))
    return sorted(results, key=lambda p: (p.start_char, p.end_char))


def lexicon_sentiment_for_text(text):
    lower = text.lower()
    words = {w.lower() for w in re.findall(r"[A-Za-z']+", text)}
    pos = len(words & POSITIVE_WORDS)
    neg = len(words & NEGATIVE_WORDS)
    for positive_word in POSITIVE_WORDS:
        if re.search(r"\b(isn't|not|never|no)\s+" + re.escape(positive_word) + r"\b", lower):
            pos = max(0, pos - 1)
            neg += 1
    if pos > neg:
        return 'positive'
    if neg > pos:
        return 'negative'
    return 'neutral'


def evaluate_aspect_extractor(extractor, reviews):
    tp = fp = fn = 0
    for review in reviews:
        gold = {(a.start_char, a.end_char, a.aspect.lower()) for a in review.annotations}
        pred = {(p.start_char, p.end_char, p.aspect.lower()) for p in extractor(review.review_text)}
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}

all_processed_reviews = splits_t05['train'] + splits_t05['dev'] + splits_t05['test']
baseline_metrics = evaluate_aspect_extractor(rule_extract_aspects, all_processed_reviews)
assert rule_extract_aspects('Amazing display but terrible battery.')[0].aspect.lower() == 'display'
assert lexicon_sentiment_for_text("The camera isn't good but the battery is terrible") == 'negative'

baseline_path = PROJECT_ROOT / 'models/baselines/classical_baselines.json'
baseline_path.parent.mkdir(parents=True, exist_ok=True)
baseline_path.write_text(json.dumps({'aspect_terms': ASPECT_TERMS, 'positive_words': sorted(POSITIVE_WORDS), 'negative_words': sorted(NEGATIVE_WORDS), 'aspect_metrics': baseline_metrics}, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('T06 baseline metrics:', baseline_metrics)

## T07 - BIO Labels and Token Alignment

This section converts character spans to BIO tags with offset-aware tokenization. It masks special tokens with `-100`, rejects overlapping spans, and provides readable debug output.

In [ ]:
def simple_tokenize_with_offsets(text):
    return [(m.group(0), m.start(), m.end()) for m in re.finditer(r"\w+|[^\w\s]", text)]


def spans_overlap(a, b):
    return not (a[1] <= b[0] or a[0] >= b[1])


def align_bio_labels(text, annotations, max_length=128):
    spans = sorted([(a.start_char, a.end_char, a.aspect) for a in annotations])
    for i in range(1, len(spans)):
        if spans_overlap(spans[i - 1], spans[i]):
            raise ValueError(f'overlapping spans: {spans[i - 1]} and {spans[i]}')
    pieces = simple_tokenize_with_offsets(text)
    if len(pieces) + 2 > max_length:
        raise ValueError('sequence would be truncated; increase max_length or handle truncation explicitly')
    tokens = ['[CLS]'] + [p[0] for p in pieces] + ['[SEP]']
    offsets = [(0, 0)] + [(p[1], p[2]) for p in pieces] + [(0, 0)]
    labels = [-100] + [BIO_LABEL_TO_ID['O']] * len(pieces) + [-100]
    for start, end, aspect in spans:
        covered = []
        for idx, (_, tok_start, tok_end) in enumerate(pieces, start=1):
            if tok_start >= start and tok_end <= end:
                covered.append(idx)
        if not covered:
            raise ValueError(f'no token coverage for span {start}-{end} {aspect!r}')
        labels[covered[0]] = BIO_LABEL_TO_ID['B-ASP']
        for idx in covered[1:]:
            labels[idx] = BIO_LABEL_TO_ID['I-ASP']
    return {'tokens': tokens, 'offset_mapping': offsets, 'labels': labels}


def debug_alignment(aligned):
    return list(zip(aligned['tokens'], aligned['offset_mapping'], aligned['labels']))

sample_review = Review(review_id='align-1', product_id='p', product_category='Laptop', review_text='Great battery life, sharp display.', annotations=[
    AspectAnnotation(aspect='battery life', sentiment='positive', start_char=6, end_char=18),
    AspectAnnotation(aspect='display', sentiment='positive', start_char=26, end_char=33),
])
aligned = align_bio_labels(sample_review.review_text, sample_review.annotations)
assert aligned['tokens'][0] == '[CLS]' and aligned['labels'][0] == -100
assert BIO_LABEL_TO_ID['B-ASP'] in aligned['labels']
assert BIO_LABEL_TO_ID['I-ASP'] in aligned['labels']
try:
    align_bio_labels('bad overlap', [AspectAnnotation(aspect='bad', sentiment='negative', start_char=0, end_char=3), AspectAnnotation(aspect='bad overlap', sentiment='negative', start_char=0, end_char=11)])
    raise AssertionError('overlap was accepted')
except ValueError:
    pass
try:
    align_bio_labels(' '.join(['token'] * 200), [], max_length=16)
    raise AssertionError('truncation was accepted')
except ValueError:
    pass
print('T07 alignment sample:', debug_alignment(aligned))

## T08 - Aspect Extractor Training Smoke

This is a lightweight smoke trainer that learns an aspect vocabulary from training annotations and saves a reloadable checkpoint manifest. It is not a replacement for the required DistilBERT training, but it lets the Kaggle notebook exercise the training/evaluation contract immediately.

In [ ]:
def train_smoke_aspect_extractor(train_reviews):
    vocab = sorted({ann.aspect.lower() for review in train_reviews for ann in review.annotations if ann.aspect.strip()})
    checkpoint = {
        'model_type': 'smoke_rule_aspect_extractor',
        'seed': 42,
        'aspect_vocabulary': vocab,
        'training_reviews': len(train_reviews),
        'training_annotations': sum(len(r.annotations) for r in train_reviews),
    }
    path = PROJECT_ROOT / 'models/aspect_extractor/smoke_aspect_extractor.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(checkpoint, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    manifest = {'checkpoint_path': str(path), 'checkpoint_sha256': sha256_file(path), **checkpoint}
    (path.parent / 'manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return manifest

aspect_manifest = train_smoke_aspect_extractor(splits_t05['train'])
assert aspect_manifest['training_reviews'] == len(splits_t05['train'])
assert len(aspect_manifest['aspect_vocabulary']) > 0
print('T08 aspect extractor manifest:', aspect_manifest)

## T09 - Aspect Extractor Evaluation and Inference

`extract_aspects` reconstructs exact character spans from the smoke aspect vocabulary, deduplicates overlapping matches, and returns contract-valid `AspectPrediction` objects.

In [ ]:
def load_smoke_aspect_terms(manifest):
    return json.loads(Path(manifest['checkpoint_path']).read_text(encoding='utf-8'))['aspect_vocabulary']

SMOKE_ASPECT_TERMS = load_smoke_aspect_terms(aspect_manifest)


def extract_aspects(text):
    predictions = []
    occupied = []
    for term in sorted(SMOKE_ASPECT_TERMS, key=len, reverse=True):
        for match in re.finditer(r'(?<!\w)' + re.escape(term) + r'(?!\w)', text, flags=re.IGNORECASE):
            span = (match.start(), match.end())
            if any(spans_overlap(span, taken) for taken in occupied):
                continue
            occupied.append(span)
            predictions.append(AspectPrediction(aspect=text[span[0]:span[1]], start_char=span[0], end_char=span[1], confidence=0.80))
    return sorted(predictions, key=lambda p: (p.start_char, p.end_char))

aspect_eval = evaluate_aspect_extractor(extract_aspects, splits_t05['test'])
for pred in extract_aspects('Amazing display but terrible battery.'):
    assert 'Amazing display but terrible battery.'[pred.start_char:pred.end_char] == pred.aspect
assert extract_aspects('No known feature mentioned here.') == []
assert aspect_eval['f1'] >= 0.70, aspect_eval
print('T09 smoke aspect eval:', aspect_eval)
print('T09 example:', [p.model_dump() for p in extract_aspects('Amazing display but terrible battery.')])

## T10 - Aspect-Sentiment Pair Builder

This section turns annotated reviews into `(review_text, aspect_text)` examples for aspect-conditioned sentiment classification while preserving IDs and offsets.

In [ ]:
def build_sentiment_pairs(reviews, max_chars=512):
    pairs = []
    seen = set()
    for review in reviews:
        for ann in review.annotations:
            aspect = ann.aspect
            text = review.review_text
            if len(text) > max_chars:
                center = max(0, ann.start_char - max_chars // 2)
                text = text[center:center + max_chars]
                if aspect.lower() not in text.lower():
                    text = (aspect + ' [SEP] ' + review.review_text[:max_chars - len(aspect) - 7]).strip()
            key = (review.review_id, ann.start_char, ann.end_char, ann.sentiment.value)
            if key in seen:
                continue
            seen.add(key)
            pairs.append({
                'review_id': review.review_id,
                'product_id': review.product_id,
                'review_text': text,
                'aspect_text': aspect,
                'start_char': ann.start_char,
                'end_char': ann.end_char,
                'label': ann.sentiment.value,
            })
    return pairs

pairs_by_split = {split: build_sentiment_pairs(rows) for split, rows in splits_t05.items()}
pair_counts = {split: len(rows) for split, rows in pairs_by_split.items()}
label_distribution = dict(Counter(pair['label'] for rows in pairs_by_split.values() for pair in rows))
assert pair_counts['train'] > 0
assert all(pair['aspect_text'].lower() in pair['review_text'].lower() for rows in pairs_by_split.values() for pair in rows)
for split, rows in pairs_by_split.items():
    write_jsonl(processed_dir / f'sentiment_pairs_{split}.jsonl', rows)
print('T10 pair counts:', pair_counts)
print('T10 label distribution:', label_distribution)

## T11 - Aspect Sentiment Training Smoke

This smoke trainer saves a deterministic aspect-conditioned sentiment model based on context-word counts. It supports the same callable shape T12 needs and can be replaced later by DistilBERT sequence classification.

In [ ]:
def words_near_aspect(text, aspect, radius=6):
    aspect_match = re.search(re.escape(aspect), text, flags=re.IGNORECASE)
    if aspect_match:
        boundaries = [0]
        for marker in re.finditer(r"\b(?:but|however|although|though)\b|[.;!]", text, flags=re.IGNORECASE):
            boundaries.append(marker.start())
            boundaries.append(marker.end())
        boundaries.append(len(text))
        boundaries = sorted(set(boundaries))
        for start, end in zip(boundaries, boundaries[1:]):
            if start <= aspect_match.start() and aspect_match.end() <= end:
                clause = text[start:end]
                clause_words = [m.group(0).lower() for m in re.finditer(r"[A-Za-z']+", clause)]
                if clause_words:
                    return clause_words
    words = [(m.group(0).lower(), m.start(), m.end()) for m in re.finditer(r"[A-Za-z']+", text)]
    if not aspect_match:
        return [w for w, _, _ in words]
    aspect_center = (aspect_match.start() + aspect_match.end()) / 2
    distances = sorted(((abs(((s + e) / 2) - aspect_center), w) for w, s, e in words), key=lambda x: x[0])
    return [w for _, w in distances[:radius]]


def train_smoke_sentiment_model(train_pairs):
    counts = {label: Counter() for label in ['positive', 'neutral', 'negative']}
    priors = Counter()
    for pair in train_pairs:
        priors[pair['label']] += 1
        for word in words_near_aspect(pair['review_text'], pair['aspect_text']):
            counts[pair['label']][word] += 1
    model = {
        'model_type': 'smoke_aspect_conditioned_sentiment',
        'seed': 42,
        'priors': dict(priors),
        'word_counts': {label: dict(counter) for label, counter in counts.items()},
        'positive_words': sorted(POSITIVE_WORDS),
        'negative_words': sorted(NEGATIVE_WORDS),
    }
    path = PROJECT_ROOT / 'models/sentiment_classifier/smoke_sentiment_model.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(model, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    manifest = {'checkpoint_path': str(path), 'checkpoint_sha256': sha256_file(path), 'training_pairs': len(train_pairs), 'label_distribution': dict(priors)}
    (path.parent / 'manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    return manifest

sentiment_manifest = train_smoke_sentiment_model(pairs_by_split['train'])
assert sentiment_manifest['training_pairs'] == len(pairs_by_split['train'])
print('T11 sentiment manifest:', sentiment_manifest)

## T12 - Sentiment Evaluation and Inference

`predict_sentiment(review, aspect)` returns a label, confidence, and probabilities for all three classes. The smoke predictor uses the requested aspect to choose a local context window, so different aspects in the same review can receive different labels.

In [ ]:
SMOKE_SENTIMENT_MODEL = json.loads(Path(sentiment_manifest['checkpoint_path']).read_text(encoding='utf-8'))


def softmaxish(scores):
    min_score = min(scores.values())
    shifted = {k: v - min_score + 1.0 for k, v in scores.items()}
    total = sum(shifted.values())
    return {k: v / total for k, v in shifted.items()}


def predict_sentiment(review, aspect):
    text = review.review_text if hasattr(review, 'review_text') else str(review)
    aspect_text = aspect.aspect if hasattr(aspect, 'aspect') else str(aspect)
    context_words = set(words_near_aspect(text, aspect_text, radius=7))
    scores = {'positive': 1.0, 'neutral': 1.0, 'negative': 1.0}
    scores['positive'] += 2.0 * len(context_words & POSITIVE_WORDS)
    scores['negative'] += 2.0 * len(context_words & NEGATIVE_WORDS)
    if not (context_words & POSITIVE_WORDS) and not (context_words & NEGATIVE_WORDS):
        scores['neutral'] += 1.0
    probs_raw = softmaxish(scores)
    label = max(probs_raw, key=probs_raw.get)
    probs = {SentimentLabel.positive: probs_raw['positive'], SentimentLabel.neutral: probs_raw['neutral'], SentimentLabel.negative: probs_raw['negative']}
    return SentimentPrediction(label=label, confidence=probs_raw[label], probabilities=probs)


def evaluate_sentiment_predictor(pairs):
    labels = ['positive', 'neutral', 'negative']
    confusion = {gold: {pred: 0 for pred in labels} for gold in labels}
    for pair in pairs:
        pred = predict_sentiment(pair['review_text'], pair['aspect_text']).label.value
        confusion[pair['label']][pred] += 1
    total = sum(sum(row.values()) for row in confusion.values())
    correct = sum(confusion[label][label] for label in labels)
    per_class = {}
    f1s = []
    for label in labels:
        tp = confusion[label][label]
        fp = sum(confusion[other][label] for other in labels if other != label)
        fn = sum(confusion[label][other] for other in labels if other != label)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[label] = {'precision': precision, 'recall': recall, 'f1': f1}
        if tp + fn:
            f1s.append(f1)
    return {'accuracy': correct / total if total else 0.0, 'macro_f1_observed_classes': sum(f1s) / len(f1s) if f1s else 0.0, 'per_class': per_class, 'confusion': confusion, 'total': total}

sentiment_eval = evaluate_sentiment_predictor(pairs_by_split['test'])
adversarial_text = 'good camera but poor battery'
camera_pred = predict_sentiment(adversarial_text, 'camera')
battery_pred = predict_sentiment(adversarial_text, 'battery')
assert camera_pred.label.value == 'positive'
assert battery_pred.label.value == 'negative'
assert abs(sum(camera_pred.probabilities.values()) - 1.0) < 1e-6
assert abs(sum(battery_pred.probabilities.values()) - 1.0) < 1e-6
assert sentiment_eval['accuracy'] >= 0.60, sentiment_eval
metrics_path = PROJECT_ROOT / 'models/sentiment_classifier/smoke_sentiment_metrics.json'
metrics_path.write_text(json.dumps(sentiment_eval, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('T12 smoke sentiment eval:', sentiment_eval)
print('T12 adversarial:', {'camera': camera_pred.model_dump(), 'battery': battery_pred.model_dump()})

## T00-T12 Evidence Summary

In [ ]:
def tree_summary(root: Path, max_depth: int = 3):
    rows = []
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        if len(rel.parts) <= max_depth:
            rows.append(('DIR ' if path.is_dir() else 'FILE') + ' ' + str(rel))
    return rows

prohibited_patterns = [
    re.compile(r'(^|/)\.env($|\.)'),
    re.compile(r'(^|/)node_modules/'),
    re.compile(r'(^|/)\.venv/'),
    re.compile(r'\.(ckpt|pt|pth|safetensors|bin)$'),
]
tracked_like_files = [p for p in PROJECT_ROOT.rglob('*') if p.is_file()]
prohibited = []
for p in tracked_like_files:
    rel = p.relative_to(PROJECT_ROOT).as_posix()
    if any(pattern.search(rel) for pattern in prohibited_patterns) and not rel.endswith('.env.example'):
        prohibited.append(rel)

summary = {
    'project_root': str(PROJECT_ROOT),
    'python_version': sys.version.split()[0],
    'created_file_count': len(tracked_like_files),
    'prohibited_artifacts_found': prohibited,
    'manifest_path': str(manifest_path),
    'dataset_validation': validation_results,
    't03_fixture_status': 'passed',
    't04_fixture_status': 'passed',
    't05_stats': stats_t05,
    't06_baseline_metrics': baseline_metrics,
    't08_aspect_manifest': aspect_manifest,
    't09_aspect_eval': aspect_eval,
    't10_pair_counts': pair_counts,
    't11_sentiment_manifest': sentiment_manifest,
    't12_sentiment_eval': sentiment_eval,
    'smoke_run_note': 'Synthetic fixture smoke run. Full T08/T11 DistilBERT training requires official data and GPU/runtime configuration.',
}

evidence_path = PROJECT_ROOT / 'audit/reports/gate-A0-A3-notebook-smoke-run.json'
evidence_path.write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + '\n', encoding='utf-8')
print('\n'.join(tree_summary(PROJECT_ROOT)))
print('\nEvidence summary:', evidence_path)
assert not prohibited, f'Prohibited artifacts found: {prohibited}'
print('T00-T12 smoke evidence checks passed.')

## Handoff

Task: T00-T12  
Commit: not applicable in Kaggle notebook context  
Files changed: generated under `PROJECT_ROOT` by this notebook  
Commands run: notebook cells above  
Results: schema/config checks, dataset manifest dry-run, T03/T04 converter fixture checks, T05 preprocessing/splits/stats, T06 baselines, T07 BIO alignment, T08/T09 smoke aspect training/eval/inference, T10 pair building, T11/T12 smoke sentiment training/eval/inference  
Fixtures/data/model versions: synthetic fixtures only; smoke model JSON manifests under `models/`; no restricted raw data or transformer checkpoints generated  
Known limitations: official datasets/checksums are required for full A1; DistilBERT model training/evaluation for A2/A3 is scaffolded as smoke logic, not completed academic training; Laptop-ACOS implicit aspect records are rejected until the schema supports implicit mentions  
Contract changes: none beyond initial T01 contract implementation  
Ready for audit gate: runnable smoke evidence through T12; not a full release/audit pass for real-data metrics  
Next tasks unblocked: T13 ontology and T14 normalization can be prototyped on fixture data; real T05-T12 audit requires official data and transformer training.